# 22: XGBoost Baseline & Epidemic Kinetic Dynamics

This notebook trains an XGBoost model optimized for non-linear epidemic kinetics (case velocity, acceleration, surge jerk, and commuter pressure).


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

print('✅ Core libraries imported successfully!')


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pure NumPy Metric Calculations (No seaborn / No sklearn.metrics)
def calc_rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def calc_mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def calc_pr_auc(y_true_binary, y_pred_scores):
    thresholds = np.sort(np.unique(y_pred_scores))[::-1]
    precisions, recalls = [1.0], [0.0]
    total_positives = np.sum(y_true_binary == 1)
    if total_positives == 0:
        return 0.0
    for th in thresholds:
        tp = np.sum((y_pred_scores >= th) & (y_true_binary == 1))
        fp = np.sum((y_pred_scores >= th) & (y_true_binary == 0))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        recall = tp / total_positives
        precisions.append(precision)
        recalls.append(recall)
    precisions.append(0.0)
    recalls.append(1.0)
    sorted_pairs = sorted(zip(recalls, precisions))
    r_sorted, p_sorted = zip(*sorted_pairs)
    try:
        return float(np.trapezoid(p_sorted, r_sorted))
    except AttributeError:
        return float(np.trapz(p_sorted, r_sorted))

def find_data_file(filename):
    possible_paths = [
        filename,
        os.path.join('.', filename),
        os.path.join('..', filename),
        os.path.join('Weikun', filename),
        os.path.join('..', 'Weikun', filename),
        os.path.join('Ayaan', filename),
        os.path.join('..', 'Ayaan', filename),
        os.path.join('nj_daily_feature_outputs', filename),
        os.path.join('Weikun', 'nj_daily_feature_outputs', filename),
        os.path.join('njny_commute_outputs', filename),
        os.path.join('Weikun', 'njny_commute_outputs', filename)
    ]
    for p in possible_paths:
        if os.path.exists(p):
            return p
    for root, dirs, files in os.walk('.'):
        if filename in files:
            return os.path.join(root, filename)
    parent_dir = os.path.abspath(os.path.join('.', '..'))
    for root, dirs, files in os.walk(parent_dir):
        if filename in files:
            return os.path.join(root, filename)
    return None

def load_or_generate_dataset():
    feat_path = find_data_file('nj_daily_features.csv.gz') or find_data_file('final_ml_feature_matrix.csv')
    commute_path = find_data_file('nj_commuter_matrix.csv')
    
    if feat_path:
        print(f"📖 Found feature matrix at: '{feat_path}'")
        df_feat = pd.read_csv(feat_path, compression='gzip' if feat_path.endswith('.gz') else None)
        df_feat['date'] = pd.to_datetime(df_feat['date'])
        df_feat['zip_code'] = df_feat['zip_code'].astype(str).str.zfill(5)
        unique_zips = sorted(df_feat['zip_code'].unique())
        N = len(unique_zips)
        zip_map = {z: i for i, z in enumerate(unique_zips)}
        if commute_path:
            print(f"🗺️ Ingesting LODES commuter matrix from: '{commute_path}'")
            w_df = pd.read_csv(commute_path, index_col=0)
            W = w_df.values
            row_sums = W.sum(axis=1, keepdims=True)
            W = np.divide(W, row_sums, out=np.zeros_like(W), where=row_sums!=0)
        else:
            np.random.seed(101)
            raw_W = np.random.uniform(0.01, 0.15, size=(N, N))
            np.fill_diagonal(raw_W, 0.0)
            W = raw_W / raw_W.sum(axis=1, keepdims=True)
        return df_feat, W, zip_map
    else:
        print("ℹ️ Cleaned feature file not found locally. Generating synthetic dataset...")
        num_zips, num_days = 20, 365
        nj_zips = [f"07{i:03d}" for i in range(100, 100 + num_zips)]
        dates = pd.date_range(start='2020-03-01', periods=num_days, freq='D')
        records = []
        np.random.seed(42)
        for z_idx, z in enumerate(nj_zips):
            base_pop = np.random.randint(25000, 85000)
            density = np.random.uniform(5000, 35000)
            svi = np.random.uniform(0.15, 0.85)
            t = np.arange(num_days)
            wave1 = 45 * np.exp(-((t - 60)**2) / (2 * 15**2))
            wave2 = 80 * np.exp(-((t - 220)**2) / (2 * 25**2))
            baseline = 5 + np.random.poisson(3, num_days)
            cases = np.maximum(0, (wave1 + wave2 + baseline) * (1 + 0.1 * z_idx)).astype(int)
            for d, c in zip(dates, cases):
                records.append({
                    'zip_code': z, 'date': d, 'cases': c, 'population': base_pop,
                    'pop_density_sqmi': density, 'svi_score': svi
                })
        df = pd.DataFrame(records).sort_values(['zip_code', 'date'])
        N = num_zips
        zip_map = {z: i for i, z in enumerate(nj_zips)}
        np.random.seed(101)
        raw_W = np.random.uniform(0.01, 0.15, size=(N, N))
        np.fill_diagonal(raw_W, 0.0)
        W = raw_W / raw_W.sum(axis=1, keepdims=True)
        df['cases_lag1'] = df.groupby('zip_code')['cases'].shift(1)
        df['cases_lag7'] = df.groupby('zip_code')['cases'].shift(7)
        df['case_velocity'] = df.groupby('zip_code')['cases'].diff().fillna(0)
        df['case_acceleration'] = df.groupby('zip_code')['case_velocity'].diff().fillna(0)
        roll7 = df.groupby('zip_code')['cases'].transform(lambda x: x.rolling(7, min_periods=1).mean())
        roll7_prev = df.groupby('zip_code')['cases'].transform(lambda x: x.shift(7).rolling(7, min_periods=1).mean())
        df['seir_R_t'] = np.where(roll7_prev > 0, roll7 / roll7_prev, 1.0)
        df['seir_R_t'] = np.clip(df['seir_R_t'], 0.2, 4.0)
        df['est_ed_visits'] = df['cases'] * 0.12
        pivot_lag1 = df.pivot(index='date', columns='zip_code', values='cases_lag1').fillna(0)
        spatial_lag_matrix = pivot_lag1.values @ W.T
        spatial_lag_df = pd.DataFrame(spatial_lag_matrix, index=pivot_lag1.index, columns=pivot_lag1.columns)
        spatial_lag_long = spatial_lag_df.reset_index().melt(id_vars='date', var_name='zip_code', value_name='spatial_lag_cases_lag1')
        df = df.merge(spatial_lag_long, on=['date', 'zip_code'], how='left')
        df['commuting_weighted_covid_pressure'] = df['spatial_lag_cases_lag1'] * 1.25
        df_clean = df.dropna().reset_index(drop=True)
        return df_clean, W, zip_map


In [ ]:
feature_df, W_matrix, zip_to_idx = load_or_generate_dataset()
feature_cols = ['cases_lag1', 'cases_lag7', 'spatial_lag_cases_lag1', 'commuting_weighted_covid_pressure', 'seir_R_t', 'case_velocity', 'case_acceleration', 'est_ed_visits', 'pop_density_sqmi', 'svi_score']
print(f'Loaded dataset: {len(feature_df)} rows.')


In [ ]:
# Fit XGBoost Baseline
X = feature_df[feature_cols].values
y = feature_df['cases'].values

xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1)
xgb_model.fit(X, y)

preds = xgb_model.predict(X)
rmse = calc_rmse(y, preds)
mae = calc_mae(y, preds)

print('=== XGBoost Baseline Model Fit ===')
print(f'RMSE: {rmse:.4f} cases')
print(f'MAE:  {mae:.4f} cases')

# Feature Importances
importances = xgb_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'XGB Importance': importances
}).sort_values('XGB Importance', ascending=False)

print('=== XGBoost Feature Importances ===')
print(importance_df.to_string(index=False))

# Plot Feature Importances
plt.figure(figsize=(10, 5))
plt.barh(importance_df['Feature'][::-1], importance_df['XGB Importance'][::-1], color='#d95f02')
plt.title('XGBoost Feature Importance Rankings (Epidemic Kinetics)')
plt.xlabel('Importance Score')
plt.tight_layout()
os.makedirs('xgb_baseline_outputs', exist_ok=True)
plt.savefig('xgb_baseline_outputs/xgb_feature_importances.png', dpi=150)
plt.close()

importance_df.to_csv('xgb_baseline_outputs/xgb_feature_importances.csv', index=False)
print('✅ Exported feature importances to xgb_baseline_outputs/xgb_feature_importances.csv')
